<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>


<p><font size="5" color='grey'> <b>
Skill Compliance
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul

# LangSmith Env-Vars VOR allen LangChain-Imports setzen
import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M99-Skill-Compliance"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---

Dieses Modul zeigt, warum **Skills** ein wichtiges Mittel zur Steuerung von Agenten sind.

Ein Skill ist kein normaler Prompt, sondern ein **wiederverwendbares Arbeitsrezept**:
- wann ein bestimmtes Vorgehen aktiviert wird
- welche Prüfschritte in welcher Reihenfolge nötig sind
- welche Regeln, Referenzen und Hilfsskripte der Agent heranzieht

Im Mittelpunkt steht hier ein **Compliance-Skill**. Er demonstriert drei Kernideen:

| Idee | Bedeutung im Agenten-Kontext |
|---|---|
| **Workflow-Orchestrierung** | Der Agent arbeitet feste Prüfschritte in Reihenfolge ab |
| **Guardrails** | Kritische Prüfungen werden nicht stillschweigend übersprungen |
| **Progressive Disclosure** | Nur die Kernlogik liegt in `SKILL.md`, Details werden bei Bedarf geladen |

**Lernziele:**
- Skills als Mittel zur Agentensteuerung einordnen
- eine gute Trennung zwischen `SKILL.md`, `references/` und `scripts/` verstehen
- einen Compliance-Skill lokal analysieren und testen


# 2 | Prompt vs. Skill
---

Ein **Prompt** gibt einem Agenten eine Anweisung für eine einzelne Aufgabe. Ein **Skill** ist ein wiederverwendbares Arbeitsrezept mit fester Struktur, Regeln und Hilfsmitteln.

| | Prompt | Skill |
|---|---|---|
| **Zweck** | Einzelne Antwort oder Aufgabe | Wiederverwendbarer Workflow |
| **Struktur** | Freitext | Definierter Ablauf mit Pflichtschritten |
| **Guardrails** | Keine oder ad-hoc | Explizit — was darf nicht übersprungen werden |
| **Referenzen** | Alles im Prompt | Ausgelagert in `references/` — nur bei Bedarf geladen |
| **Hilfsskripte** | Keine | `scripts/` für deterministische Teilaufgaben |
| **Wiederverwendung** | Copy-Paste | Versioniert, referenzierbar, updatebar |
| **Typischer Einsatz** | Frage beantworten, Text generieren | Compliance, Genehmigung, Onboarding, Review |



**Faustregel:** Sobald ein Prozess Pflichtschritte, Eskalationsregeln oder eine dokumentierte Entscheidung erfordert — Skill statt Prompt.

# 3 | Skill-Design für Agenten
---

<p><font color='black' size="5">Warum gehört das in einen Agenten-Kurs?</font></p>

Skills operationalisieren Fachwissen. Der Agent bekommt dadurch nicht nur Antworten, sondern eine **strukturierte Handlungslogik**.

Gerade bei Compliance-, Security- oder Freigabeprozessen reicht ein freier Prompt oft nicht aus. Man will stattdessen:
- Pflichtprüfungen definieren
- Eskalationsregeln festlegen
- Entscheidungsdokumentation erzwingen
- wiederkehrende Logik verlässlich wiederverwenden

<p><font color='black' size="5">Empfohlene Aufteilung</font></p>

| Datei-/Ordner-Typ | Rolle |
|---|---|
| `SKILL.md` | Kernablauf, Trigger, Guardrails, Verweise |
| `references/*.md` | Detailwissen, Regeln, Checklisten, Beispiele |
| `scripts/` | deterministische Hilfslogik für fragile oder wiederkehrende Schritte |

Das ist genau der Punkt, den Ihre ursprüngliche Frage adressiert hat: **Ja, Skills dürfen detailliert sein, aber die Details sollten sauber aufgeteilt werden.**

In [ ]:
import urllib.request

GITHUB_BASE = "https://raw.githubusercontent.com/ralf-42/Agenten/main/06_skill/compliance"

def fetch_skill_file(path: str) -> str:
    url = f"{GITHUB_BASE}/{path}"
    with urllib.request.urlopen(url) as r:
        return r.read().decode("utf-8")

# Überblick über verfügbare Dateien im Skill
skill_files = [
    "SKILL.md",
    "references/checklist.md",
    "references/risk_rules.md",
    "references/examples.md",
    "scripts/assess_risk.py",
]
print(f"Skill-Quelle: {GITHUB_BASE}\n")
for f in skill_files:
    print(f"  {f}")

# 4 | Hands-On: Compliance-Skill lesen
---

In [ ]:
skill_text = fetch_skill_file("SKILL.md")
print(skill_text[:3000])

Achten Sie beim Lesen auf drei Dinge:
- **Trigger-Beschreibung:** Wann soll der Skill aktiv werden?
- **Core Workflow:** Welche Schritte sind verpflichtend?
- **Guardrails:** Was darf der Agent nicht behaupten oder überspringen?


In [ ]:
for name in ['checklist.md', 'risk_rules.md', 'examples.md']:
    content = fetch_skill_file(f"references/{name}")
    mprint(f'\n### {name}\n')
    mprint(content[:1500])

# 5 | Hands-On: deterministische Risiko-Prüfung
---

Der Skill enthält zusätzlich ein kleines Skript. Das ist nützlich, wenn ein Teil der Logik **nicht nur beschrieben, sondern zuverlässig berechnet** werden soll.

In [ ]:
import json

# assess_risk.py aus GitHub laden — in eigenem Namespace ausführen,
# damit der if __name__ == "__main__" Block nicht getriggert wird
script_code = fetch_skill_file("scripts/assess_risk.py")
_ns = {"__name__": "__exec__"}
exec(script_code, _ns)
assess = _ns["assess"]  # Funktion in den Notebook-Scope holen

case = {
    'subject_type': 'vendor',
    'subject_name': 'Example Trading LLC',
    'country': 'RU',
    'transaction_amount': 18000,
    'sanctions_hit': False,
    'adverse_media_hit': True,
    'pep_flag': False,
    'documents_complete': True,
}

risk_result = assess(case)
print(json.dumps(risk_result, indent=2))

In [ ]:
from textwrap import dedent

decision_note = dedent(f'''
### Compliance Decision

- Case: {case['subject_type']} for {case['subject_name']}
- Checks performed: sanctions screening, geography review, transaction size review, documentation check
- Risk level: {risk_result['risk_level']}
- Decision: {risk_result['decision']}
- Rationale: {', '.join(risk_result['reasons'])}
- Missing information or escalation point: none in this training example
- Audit note: simulated training assessment, no live sanctions source connected
''').strip()

mprint(decision_note)

# 6 | Agent mit Compliance-Skill
---

Bisher haben wir den Skill manuell gelesen und das Skript direkt aufgerufen. Jetzt übergeben wir beides an einen Agenten:

- **`SKILL.md`** wird als System-Prompt geladen — der Agent kennt damit den vollständigen Compliance-Workflow
- **`assess_risk`** wird als `@tool` bereitgestellt — der Agent kann die deterministische Prüfung selbst aufrufen
- Der Agent entscheidet eigenständig, wann er das Tool einsetzt und wie er das Ergebnis kommuniziert

| Komponente | Rolle |
|---|---|
| System-Prompt (SKILL.md) | Steuert Verhalten, Pflichtschritte, Guardrails |
| `@tool assess_risk` | Deterministische Risikobewertung |
| Agent | Orchestriert beides, erzeugt Compliance Decision |

In [ ]:
from langchain_core.tools import tool

@tool
def compliance_check(
    subject_type: str,
    subject_name: str,
    country: str,
    transaction_amount: float = 0,
    sanctions_hit: bool = False,
    adverse_media_hit: bool = False,
    pep_flag: bool = False,
    documents_complete: bool = True,
) -> str:
    """Führt eine deterministische Compliance-Risikoprüfung durch und gibt
    risk_score, risk_level, decision und reasons zurück."""
    case = {
        "subject_type": subject_type,
        "subject_name": subject_name,
        "country": country,
        "transaction_amount": transaction_amount,
        "sanctions_hit": sanctions_hit,
        "adverse_media_hit": adverse_media_hit,
        "pep_flag": pep_flag,
        "documents_complete": documents_complete,
    }
    return json.dumps(assess(case), indent=2)

print("Tool registriert:", compliance_check.name)

In [ ]:
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# SKILL.md als System-Prompt laden
skill_system_prompt = fetch_skill_file("SKILL.md")

llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)
memory = MemorySaver()

agent = create_react_agent(
    model=llm,
    tools=[compliance_check],
    prompt=skill_system_prompt,
    checkpointer=memory,
)

# Thread-ID definiert die Konversationssession
session = {"configurable": {"thread_id": "compliance-demo-1"}}

print("Agent erstellt mit create_react_agent + MemorySaver")

In [ ]:
import time

# Neue Session für jeden Testlauf (verhindert Überlappung mit vorherigen Runs)
session = {"configurable": {"thread_id": f"compliance-{int(time.time())}"}}

# ── Aufruf 1: vollständiger Fließtext — sanctions_clearance_confirmed fehlt ──
user_input_1 = (
    "Wir wollen einen neuen Lieferanten onboarden: "
    "Example Trading LLC aus Russland. "
    "Transaktionsvolumen 18.000 EUR. "
    "Geschäftszweck: Einkauf von Industriekomponenten. "
    "Zahlungskontext: reguläre Überweisung nach Wareneingang. "
    "Es gibt Hinweise auf negative Medienberichte. "
    "Alle Dokumente liegen vor."
)

response_1 = agent.invoke(
    {"messages": [("human", user_input_1)]},
    config=session,
)
mprint("### 1. Aufruf")
mprint("---")
mprint(response_1["messages"][-1].content)

⬆️ **Guardrail aktiv:** Alle anderen Felder extrahiert der Agent aus dem Fließtext. Nur `sanctions_clearance_confirmed` fragt er explizit nach — dieses Feld kann nicht inferiert werden und ist im SKILL.md als nicht-ableitbar markiert.

Der User bestätigt jetzt die Sanctions-Prüfung — der Agent hat den vollen Kontext durch das Memory:

In [ ]:
# ── Aufruf 2: User bestätigt Sanctions-Prüfung — gleiche session! ────────────
user_input_2 = "Die Sanctions-Prüfung wurde formal durchgeführt, kein Treffer."

response_2 = agent.invoke(
    {"messages": [("human", user_input_2)]},
    config=session,       # ← selbe session wie Aufruf 1
)

mprint("### 2. Aufruf")
mprint("---")
mprint(response_2["messages"][-1].content)

In [ ]:
show_trace("M99-Skill-Compliance", show_steps=True)

# 7 | Fazit
---

Dieses Beispiel zeigt die saubere Rollenverteilung eines Skills:
- `SKILL.md` steuert das Verhalten des Agenten
- `references/` halten Fachregeln und Beispiele aus dem Kernkontext heraus
- `scripts/` übernehmen deterministische Teilaufgaben

Genau deshalb passt das Thema in den Kurs **Agenten**: Es geht nicht nur um Antworten, sondern um **zuverlässig gesteuerte agentische Prozesse**.

# A | Aufgabe
---



<p><font color='black' size="5">
Skill erweitern
</font></p>

Erweitern Sie den Compliance-Skill um einen zweiten Entscheidungspfad, zum Beispiel für Lieferanten-Onboarding oder Auszahlungsfreigaben.

**Teilaufgaben:**
- Ergänzen Sie mindestens eine neue Regel in `references/risk_rules.md`
- Fügen Sie ein weiteres Beispiel in `references/examples.md` hinzu
- Passen Sie das Skript so an, dass die neue Regel in die Bewertung einfließt
